# M9 — Matching v2 diagnostics + spot-check

Validation of `build_matched_dataset.py` v2 output.

**What this notebook does:**
1. Layer distribution (counts + % of NKC corpus)
2. SCKN positive coverage by Layer
3. Distribution of `fuzzy_score` in Layers 2/3 (sanity-check threshold pin)
4. **60 visual spot-checks** — 20 random matches per Layer with Goodreads URLs.
   Open each link, compare GR title+author to NKC fields. Mark ✓ / ✗ / ? in `verdict` column.
5. **30 unmatched audit** — random NKC records that fell through the cascade, to
   understand whether they're (a) not on Goodreads, (b) missing `original_title`,
   or (c) author transliteration we can't fix.
6. **Positives browser** — all training-set positives (`sckn_appearances >= 1`) in
   one table so you can scan for label noise.

**Pass criteria:**
- ≤ 2 wrong-book matches out of 20 in each Layer (≤10%). More than that in Layer 2 or 3
  → raise thresholds and rerun `build_matched_dataset.py`.
- No obvious wrong-book matches among the positives.

Random seed is pinned (`SEED = 42`) so re-running yields the same spot-check set.

In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

SEED = 42
rng = np.random.default_rng(SEED)

REPO    = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM = REPO / "data" / "interim"

print(f"REPO    : {REPO}")
print(f"INTERIM : {INTERIM}")

REPO    : /home/firstone/Bachelors-thesis
INTERIM : /home/firstone/Bachelors-thesis/data/interim


In [4]:
matched = pd.read_csv(INTERIM / "matched_dataset.csv", dtype=str, keep_default_na=False)
training = pd.read_csv(INTERIM / "training_dataset.csv", dtype=str, keep_default_na=False)

print(f"matched_dataset.csv  : {len(matched):,} rows, {len(matched.columns)} cols")
print(f"training_dataset.csv : {len(training):,} rows, {len(training.columns)} cols")
print()
print("Columns:")
for c in matched.columns:
    print(f"  {c}")

matched_dataset.csv  : 225,483 rows, 29 cols
training_dataset.csv : 26,090 rows, 29 cols

Columns:
  nkc_id
  oclc
  czech_isbn
  czech_title
  original_title
  original_isbn
  original_sysnum
  author
  secondary_authors
  czech_pub_year
  source_lang
  genres
  match_layer
  matched_book_id
  gr_work_id
  fuzzy_score
  gr_title
  gr_pub_year
  gr_ratings_count
  gr_average_rating
  gr_text_reviews_count
  gr_popular_shelves
  gr_language_code
  gr_is_ebook
  sckn_appearances
  sckn_first_year
  sckn_best_rank
  sckn_categories
  sckn_bestseller


## 1 — Layer distribution

In [5]:
layer_counts = matched["match_layer"].value_counts()
layer_pct = (layer_counts / len(matched) * 100).round(2)

summary = pd.DataFrame({
    "count": layer_counts,
    "pct_of_NKC": layer_pct.astype(str) + "%",
})
summary.index.name = "layer"
print(summary.to_string())
print()
print(f"Matched total : {len(matched) - layer_counts.get('unmatched', 0):,} "
      f"({100 - layer_pct.get('unmatched', 0):.1f}% of NKC)")

            count pct_of_NKC
layer                       
unmatched  169500     75.17%
1           39376     17.46%
2           13265      5.88%
3            3342      1.48%

Matched total : 55,983 (24.8% of NKC)


## 2 — SCKN positives by Layer

How many positives lie in each Layer? If Layer 2/3 have disproportionately few positives → those layers may be matching wrong books (real positives get diverted into wrong-book matches that don't carry SCKN labels).

But note: positive count alone doesn't prove correctness — SCKN labels join on `czech_isbn`, not `matched_book_id`, so wrong-book matches still carry the correct label. The signal here is *coverage*, not precision.

In [6]:
matched["sckn_appearances"] = pd.to_numeric(matched["sckn_appearances"], errors="coerce").fillna(0).astype(int)
matched["is_positive"] = (matched["sckn_appearances"] >= 1).astype(int)

by_layer = matched.groupby("match_layer").agg(
    n=("nkc_id", "count"),
    n_positive=("is_positive", "sum"),
    pct_positive=("is_positive", "mean"),
)
by_layer["pct_positive"] = (by_layer["pct_positive"] * 100).round(2)
by_layer = by_layer.sort_values("n", ascending=False)
print(by_layer.to_string())
print()
print(f"Total SCKN positives in matched_dataset : {matched['is_positive'].sum():,}")

                  n  n_positive  pct_positive
match_layer                                  
unmatched    169500        1609          0.95
1             39376        1320          3.35
2             13265         305          2.30
3              3342         152          4.55

Total SCKN positives in matched_dataset : 3,386


## 3 — fuzzy_score distribution for Layers 2/3

Layer 1 has `fuzzy_score = 100` by definition (exact match), so we skip it.

For Layer 2 (threshold ≥ 88) and Layer 3 (≥ 85), look at the distribution. If most scores hover near the threshold, our cutoff is barely catching anything — probably too low or too high. If scores cluster near 100, the layer is doing real work.

In [7]:
matched["fuzzy_score"] = pd.to_numeric(matched["fuzzy_score"], errors="coerce").fillna(0).astype(int)

for layer in ["2", "3"]:
    scores = matched.loc[matched["match_layer"] == layer, "fuzzy_score"]
    if len(scores) == 0:
        print(f"Layer {layer}: no matches")
        continue
    print(f"Layer {layer} — fuzzy_score distribution (n={len(scores):,})")
    print(scores.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_string())
    # Bins for histogram-like view in text
    bins = [85, 88, 90, 92, 94, 96, 98, 100, 101]
    counts, _ = np.histogram(scores, bins=bins)
    print("\n  Histogram:")
    for lo, hi, c in zip(bins[:-1], bins[1:], counts):
        bar = "█" * int(c / max(counts) * 30) if max(counts) else ""
        print(f"  [{lo:>3}-{hi-1:>3}] {c:>6,}  {bar}")
    print()

Layer 2 — fuzzy_score distribution (n=13,265)
count    13265.0
mean        99.2
std          2.3
min         88.0
10%         97.0
25%        100.0
50%        100.0
75%        100.0
90%        100.0
max        100.0

  Histogram:
  [ 85- 87]      0  
  [ 88- 89]    203  
  [ 90- 91]    218  
  [ 92- 93]    228  
  [ 94- 95]    283  
  [ 96- 97]    578  █
  [ 98- 99]    181  
  [100-100] 11,574  ██████████████████████████████

Layer 3 — fuzzy_score distribution (n=3,342)
count    3342.0
mean       95.4
std         5.0
min        85.0
10%        88.0
25%        92.0
50%        96.0
75%       100.0
90%       100.0
max       100.0

  Histogram:
  [ 85- 87]    326  ██████
  [ 88- 89]    203  ████
  [ 90- 91]    207  ████
  [ 92- 93]    253  █████
  [ 94- 95]    651  █████████████
  [ 96- 97]    217  ████
  [ 98- 99]     11  
  [100-100]  1,474  ██████████████████████████████



## 4 — Spot-check: 60 random matches (20 per Layer)

**Procedure:**
1. Open `gr_url` in a new tab.
2. Compare Goodreads page title+author with NKC `original_title` and `author`.
3. Verdict:
   - ✓ correct: same book, same author, possibly different edition
   - ✗ wrong: different book, or wrong author
   - ? unsure: ambiguous (e.g. anthology, translation issue)
4. Write your verdict in a notepad — the table is too wide to mark inline. After looking at all 60, count wrong-book matches per layer.

**Pass criteria: ≤ 2 wrong-book matches out of 20 in each Layer.**

In [8]:
def sample_layer(df: pd.DataFrame, layer: str, n: int = 20) -> pd.DataFrame:
    pool = df[df["match_layer"] == layer]
    if len(pool) == 0:
        return pool
    idx = rng.choice(len(pool), size=min(n, len(pool)), replace=False)
    sampled = pool.iloc[idx].copy()
    sampled["gr_url"] = "https://www.goodreads.com/book/show/" + sampled["matched_book_id"]
    return sampled


SPOT_COLS = [
    "match_layer", "fuzzy_score",
    "author", "original_title",
    "czech_pub_year", "source_lang",
    "matched_book_id", "gr_work_id", "gr_url",
]

In [9]:
print("=" * 90)
print("LAYER 1 — exact author + exact title (expect ≥ 95% precision)")
print("=" * 90)
spot_l1 = sample_layer(matched, "1", 20)
spot_l1[SPOT_COLS]

LAYER 1 — exact author + exact title (expect ≥ 95% precision)


,match_layer,fuzzy_score,author,original_title,czech_pub_year,source_lang,matched_book_id,gr_work_id,gr_url
176171,1,100,"Mayle, Peter",Diamond caper,2016,eng,24848332,44070875,https://www.goodreads.com/book/show/24848332
168218,1,100,"Ray, Nick",Cambodia,2015,eng,182758,176615,https://www.goodreads.com/book/show/182758
66811,1,100,"O'Brien, Kate",Last of summer,1958,eng,2368848,2375674,https://www.goodreads.com/book/show/2368848
216742,1,100,"Springer, Nancy",Case of the gypsy good-bye,2024,eng,15736348,6854559,https://www.goodreads.com/book/show/15736348
185268,1,100,"Abnett, Dan",Ravenor returned,2018,eng,1052308,1038786,https://www.goodreads.com/book/show/1052308
174217,1,100,"Fletcher, Nichola",Meat cookbook,2016,eng,21483627,40810318,https://www.goodreads.com/book/show/21483627
32795,1,100,"Pope, Alexander",Rape of the Lock,1998,eng,954434,6423297,https://www.goodreads.com/book/show/954434
21994,1,100,"Sue, Eugène",Mystères de Paris,1970,fre,631010,617313,https://www.goodreads.com/book/show/631010
159095,1,100,"Blædel, Sara",Kald mig prinsesse,2013,dan,22752193,15120815,https://www.goodreads.com/book/show/22752193
138304,1,100,"Wood, Patricia",Lottery,2009,eng,759175,2549658,https://www.goodreads.com/book/show/759175


In [10]:
print("=" * 90)
print("LAYER 2 — exact author + fuzzy title ≥ 88 (the risky one)")
print("=" * 90)
spot_l2 = sample_layer(matched, "2", 20)
spot_l2[SPOT_COLS]

LAYER 2 — exact author + fuzzy title ≥ 88 (the risky one)


,match_layer,fuzzy_score,author,original_title,czech_pub_year,source_lang,matched_book_id,gr_work_id,gr_url
169180,2,100,"Semler, Ricardo",Maverick!,2015,eng,32994,100754,https://www.goodreads.com/book/show/32994
178473,2,100,"Bridges, Jerry",Trusting God,2016,eng,9871622,123695,https://www.goodreads.com/book/show/9871622
26788,2,100,"Simpson, Dorothy","Six Feet Under, Puppet for a Corpse",1989,eng,193710,187338,https://www.goodreads.com/book/show/193710
197480,2,100,"Laloux, Frédéric",Reinventing organizations,2020,eng,20776174,40126556,https://www.goodreads.com/book/show/20776174
140815,2,100,"Moro, Javier",Sari rojo,2010,spa,6380302,6568195,https://www.goodreads.com/book/show/6380302
180799,2,100,"Millman, Dan",Hidden school,2017,eng,33107352,53688945,https://www.goodreads.com/book/show/33107352
171977,2,100,"Weisman, Alan",Countdown,2015,eng,17332183,24035802,https://www.goodreads.com/book/show/17332183
81411,2,100,"Chalker, Jack L",Wonderland gambit. The march hare network,,eng,584393,571246,https://www.goodreads.com/book/show/584393
137663,2,95,"Keyes, J. Gregory",Star Wars - the new Jedi order. Edge of victory II. Rebirth,2009,eng,35429,72183,https://www.goodreads.com/book/show/35429
90822,2,97,"Ibsen, Henrik",Byggmester Solness,1930,nor,25196157,3106807,https://www.goodreads.com/book/show/25196157


In [11]:
print("=" * 90)
print("LAYER 3 — fuzzy author ≥ 85 + exact title (smallest layer, riskiest profile)")
print("=" * 90)
spot_l3 = sample_layer(matched, "3", 20)
spot_l3[SPOT_COLS]

LAYER 3 — fuzzy author ≥ 85 + exact title (smallest layer, riskiest profile)


,match_layer,fuzzy_score,author,original_title,czech_pub_year,source_lang,matched_book_id,gr_work_id,gr_url
16893,3,100,"Durrell, Gerald Malcolm",Three singles to adventure,1995,eng,33957923,1824164,https://www.goodreads.com/book/show/33957923
83432,3,100,"Rosny, J.-H",Xipéhuz,1906,fre,34094401,55112026,https://www.goodreads.com/book/show/34094401
157577,3,94,"Fitzgerald, Francis Scott",Diamond as big as the Ritz,2013,eng,16099210,25329138,https://www.goodreads.com/book/show/16099210
179295,3,94,"Bogdan, D. L",Rivals in the Tudor court,2017,eng,9353896,14237290,https://www.goodreads.com/book/show/9353896
105683,3,100,"Asprin, Robert",Something M.Y.T.H. Inc,2003,eng,74295,3103467,https://www.goodreads.com/book/show/74295
196166,3,100,"Cornwell, Patricia Daniels",Chaos,2020,eng,28959361,49186149,https://www.goodreads.com/book/show/28959361
183337,3,92,"Reynolds, Josh",Serpent queen,2017,eng,19741603,27794792,https://www.goodreads.com/book/show/19741603
68951,3,100,"Booher, Dianna Daniels",Communicate with confidence!: how to say it right the first time and every time,1999,eng,11451776,1237730,https://www.goodreads.com/book/show/11451776
150351,3,96,"Martin, George R. R",Storm of swords,2011,eng,13253102,1164465,https://www.goodreads.com/book/show/13253102
32008,3,94,"Mueffling, Didi von",50 most romantic things ever done,1998,eng,1593281,1586347,https://www.goodreads.com/book/show/1593281


### Inline verdict columns

Re-run the cell below after each spot-check pass with your verdicts filled in. Tally then read out the wrong counts.

In [12]:
# After visually inspecting, replace the 20 placeholders for each layer with one of:
#   "✓"  — correct match
#   "✗"  — wrong-book match
#   "?"  — unsure / ambiguous
#
# Then re-run this cell to print the tally.

verdicts_l1 = ["✓"] * 20
verdicts_l2 = ["✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✓", "✗"]
verdicts_l3 = ["✓"] * 20


def tally(label: str, verdicts: list[str]) -> None:
    n = len(verdicts)
    correct = sum(1 for v in verdicts if v == "✓")
    wrong   = sum(1 for v in verdicts if v == "✗")
    unsure  = sum(1 for v in verdicts if v == "?")
    pass_   = "PASS" if wrong <= 2 else "FAIL — tighten threshold"
    print(f"{label}: correct={correct:>2}  wrong={wrong:>2}  unsure={unsure:>2}  n={n}  → {pass_}")

tally("Layer 1", verdicts_l1)
tally("Layer 2", verdicts_l2)
tally("Layer 3", verdicts_l3)

Layer 1: correct=20  wrong= 0  unsure= 0  n=20  → PASS
Layer 2: correct=19  wrong= 1  unsure= 0  n=20  → PASS
Layer 3: correct=20  wrong= 0  unsure= 0  n=20  → PASS


## 5 — Unmatched audit (30 random)

75% of NKC records didn't match anything. The question is: is that a data ceiling (book genuinely not on Goodreads) or could a smarter matcher catch more?

For each row, classify mentally:
- **A. No original_title** — `original_title` empty → cascade can't run. Real loss; would need new Goodreads side index by Czech title (poor signal).
- **B. Obscure book** — title and author exist but the book is not on Goodreads (regional textbook, government publication, niche Slovak/Polish work). Data ceiling, accept.
- **C. Transliteration mismatch** — Cyrillic author appears in NKC as one transliteration variant, Goodreads has another (e.g. "Лунача́рский" → NKC has "Lunačarskij", Goodreads has "Lunacharsky"). Fixable in principle with smarter transliteration but not in this thesis scope.
- **D. Stale GR dump** — book published in Czech post-2017 may not exist in the 2017 GR snapshot.

In [13]:
unmatched = matched[matched["match_layer"] == "unmatched"]
print(f"Unmatched pool: {len(unmatched):,} records")
print()

# Coverage breakdown of the unmatched pool — what's missing?
no_title    = (unmatched["original_title"] == "").sum()
no_author   = (unmatched["author"] == "").sum()
no_both     = ((unmatched["original_title"] == "") & (unmatched["author"] == "")).sum()
have_both   = ((unmatched["original_title"] != "") & (unmatched["author"] != "")).sum()

print("Why unmatched? (categories overlap)")
print(f"  Missing original_title       : {no_title:>7,}  ({no_title/len(unmatched):.1%})")
print(f"  Missing author               : {no_author:>7,}  ({no_author/len(unmatched):.1%})")
print(f"  Missing both                 : {no_both:>7,}  ({no_both/len(unmatched):.1%})")
print(f"  Have both (cascade ran, no hit): {have_both:>7,}  ({have_both/len(unmatched):.1%})")

Unmatched pool: 169,500 records

Why unmatched? (categories overlap)
  Missing original_title       :  84,804  (50.0%)
  Missing author               :  26,389  (15.6%)
  Missing both                 :  18,562  (11.0%)
  Have both (cascade ran, no hit):  76,869  (45.4%)


In [14]:
# Sample 30 unmatched records that have both author AND original_title — these
# are the most informative (cascade actually ran, found nothing).
audit_pool = unmatched[(unmatched["original_title"] != "") & (unmatched["author"] != "")]
idx = rng.choice(len(audit_pool), size=30, replace=False)
audit = audit_pool.iloc[idx].copy()

AUDIT_COLS = [
    "author", "original_title", "czech_title",
    "source_lang", "czech_pub_year", "oclc",
]
audit[AUDIT_COLS]

,author,original_title,czech_title,source_lang,czech_pub_year,oclc
152852,"Beurenmeister, Corina","Komm, wir gehen in den ZOO!",Zvířátka v ZOO,ger,2012,796019870
161298,"Hakvoort, Emmerich",U-boat war 1914-1918,Zlověstné oceány,eng,2013,864849393
170169,"Vela, Marcelo F",Practical manual of gastroesophageal reflux disease,Refluxní choroba jícnu - GERD,eng,2015,920663262
4700,"Burovik, Kim Aleksejevič",Rodoslovnaja veščej,Osudy věcí kolem nás,rus,1990,39429609
143453,"Schmidt, Mathias R",Nordic Fitness,Nordic fitness,ger,2010,681497181
197640,"Putz, Erna",Franz Jägerstätter - Märtyrer,Sedlák proti Hitlerovi,ger,2020,1226361881
105049,"Breedlove, Greta",Herbal home spa,Bylinkové domácí lázně,eng,2006,85717716
24568,"Volkogonov, Dmitrij Antonovič",Metodologija idejnogo vospitanija,Metodologie ideové výchovy,rus,1981,85280196
14205,"Kraft, Robert",Goldschiff und Vulkan,Vzestup a pád,ger,1997,37249942
224518,"Lagercrantz, David",Post mortem,Post mortem,swe,2025,1572322784


## 6 — Positives browser (all 1,405 training positives)

Sanity-check: scan the table below for obvious wrong-book matches that happened to share a `czech_isbn` with an SCKN entry. If you spot any, the GR URL is included.

(Project instruction: "label threshold is `sckn_appearances >= 1`" — that's exactly what defines positive here.)

In [15]:
training["sckn_appearances"] = pd.to_numeric(training["sckn_appearances"], errors="coerce").fillna(0).astype(int)
training["fuzzy_score"] = pd.to_numeric(training["fuzzy_score"], errors="coerce").fillna(0).astype(int)

positives = training[training["sckn_appearances"] >= 1].copy()
positives["gr_url"] = "https://www.goodreads.com/book/show/" + positives["matched_book_id"]
positives = positives.sort_values("sckn_appearances", ascending=False)

print(f"Positives: {len(positives):,} rows")
print(f"  by layer:")
for layer, grp in positives.groupby("match_layer"):
    print(f"    Layer {layer}: {len(grp):,}  (mean sckn_appearances = {grp['sckn_appearances'].mean():.1f})")
print()

POS_COLS = [
    "match_layer", "fuzzy_score",
    "author", "original_title", "czech_title",
    "sckn_appearances", "sckn_best_rank", "sckn_first_year", "sckn_categories",
    "czech_pub_year", "source_lang",
    "gr_url",
]
positives[POS_COLS].head(30)

Positives: 1,405 rows
  by layer:
    Layer 1: 1,082  (mean sckn_appearances = 5.4)
    Layer 2: 236  (mean sckn_appearances = 4.2)
    Layer 3: 87  (mean sckn_appearances = 10.3)



,match_layer,fuzzy_score,author,original_title,czech_title,sckn_appearances,sckn_best_rank,sckn_first_year,sckn_categories,czech_pub_year,source_lang,gr_url
7621,1,100,"Kinney, Jeff",Diary of a wimpy kid,Deník malého poseroutky,258,1,2009,literatura pro děti a mládež,2009,eng,https://www.goodreads.com/book/show/7545951
2315,1,100,"Saint-Exupéry, Antoine de",Petit prince,Malý princ,208,1,2006,literatura pro děti a mládež,2005,fre,https://www.goodreads.com/book/show/8848
15735,1,100,"Smith, Keri",Wreck this journal,Destrukční deník,139,1,2014,literatura pro děti a mládež,2014,eng,https://www.goodreads.com/book/show/18047871
909,1,100,"Brown, Dan",Da Vinci code,Šifra mistra Leonarda,111,1,2004,beletrie,2003,eng,https://www.goodreads.com/book/show/85266
20685,2,100,"Manson, Mark",Subtle art of not giving a f*ck,"Důmyslné umění, jak mít všechno u pr**le",99,1,2018,naučná literatura,2017,eng,https://www.goodreads.com/book/show/28257707
1514,2,100,"Paolini, Christopher","Eragon, Inheritance. Book one",Odkaz Dračích jezdců,98,1,2004,literatura pro děti a mládež,2004,eng,https://www.goodreads.com/book/show/7664041
12588,1,100,"Walliams, David",Gangsta granny,Babička drsňačka,94,1,2012,literatura pro děti a mládež,2012,eng,https://www.goodreads.com/book/show/12727215
1331,3,95,"Rowling, J. K",Harry Potter and the Order of the Phoenix,Harry Potter a Fénixův řád,92,1,2004,literatura pro děti a mládež,2004,eng,https://www.goodreads.com/book/show/2
5911,1,100,"Byrne, Rhonda",Secret,Tajemství,76,1,2008,naučná literatura,2008,eng,https://www.goodreads.com/book/show/598162
9228,1,100,"Kinney, Jeff",Last straw,Deník malého poseroutky,75,2,2010,literatura pro děti a mládež,2010,eng,https://www.goodreads.com/book/show/7516186


In [16]:
# Drill-down: positives in Layer 2/3 specifically (these are riskiest — fuzzy match
# could be wrong-book AND happen to share a czech_isbn that hits SCKN, producing a
# false-positive at the label-attribution level).
fuzzy_positives = positives[positives["match_layer"].isin(["2", "3"])]
print(f"Fuzzy positives (Layer 2 or 3): {len(fuzzy_positives):,}")
print()
fuzzy_positives[POS_COLS]

Fuzzy positives (Layer 2 or 3): 323



,match_layer,fuzzy_score,author,original_title,czech_title,sckn_appearances,sckn_best_rank,sckn_first_year,sckn_categories,czech_pub_year,source_lang,gr_url
20685,2,100,"Manson, Mark",Subtle art of not giving a f*ck,"Důmyslné umění, jak mít všechno u pr**le",99,1,2018,naučná literatura,2017,eng,https://www.goodreads.com/book/show/28257707
1514,2,100,"Paolini, Christopher","Eragon, Inheritance. Book one",Odkaz Dračích jezdců,98,1,2004,literatura pro děti a mládež,2004,eng,https://www.goodreads.com/book/show/7664041
1331,3,95,"Rowling, J. K",Harry Potter and the Order of the Phoenix,Harry Potter a Fénixův řád,92,1,2004,literatura pro děti a mládež,2004,eng,https://www.goodreads.com/book/show/2
1013,3,95,"Rowling, J. K",Harry Potter and the philosopher's stone,Harry Potter a kámen mudrců,70,1,2017,literatura pro děti a mládež,2017,eng,https://www.goodreads.com/book/show/35604481
12883,3,94,"James, E. L",Fifty shades of grey,Padesát odstínů šedi =,63,1,2012,beletrie,2012,eng,https://www.goodreads.com/book/show/13536858
...,...,...,...,...,...,...,...,...,...,...,...,...
38,3,100,"Smith, Wilbur A",Shout at the devil,Volání na ďábla,1,6,2003,beletrie,2003,eng,https://www.goodreads.com/book/show/906608
275,2,100,"Ruiz, Miguel",Mastery of love,"Láska, vztahy a přátelství",1,9,2004,naučná literatura,2004,eng,https://www.goodreads.com/book/show/315350
13900,2,100,"Lampard, Frank",Frankie's magic footbal - Frankie vs the pirate pillagers,Frankův kouzelný fotbal,1,9,2013,literatura pro děti a mládež,2013,eng,https://www.goodreads.com/book/show/18527473
13695,3,92,"Young, William P",Cross roads,Křižovatky,1,9,2013,beletrie,2013,eng,https://www.goodreads.com/book/show/17043092


## 7 — Decision

Based on the spot-check tallies above:

| Layer | wrong/20 | Action |
|---|---|---|
| 1 | … | Should always be 0–1. If ≥ 3, our normalize() has a bug. |
| 2 | … | ≤ 2 → keep threshold 88. ≥ 3 → raise to 92 in `build_matched_dataset.py`. |
| 3 | … | ≤ 2 → keep threshold 85. ≥ 3 → raise to 90 OR drop the layer. |

**If all three layers pass:** report numbers to assistant, proceed to Step 5 (re-run `aggregate_reviews.py`).

**If a layer fails:** report which one and the wrong-count. Assistant raises the threshold and you re-run `build_matched_dataset.py`.